<a href="https://colab.research.google.com/github/naamasarshalom-art/segmentation_cellpose/blob/main/4_model1_training.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Model 1 — Quality Control Classifier (Classifiable vs Trash)

This notebook trains **Model 1**: a binary SVM classifier that separates
classifiable nuclei (good + invaginated + unclassifiable) from trash/artifacts.

**Input:** RADIO embeddings computed in notebook `3_RADIO_embeddings.ipynb`

**Pipeline:**
1. Load embeddings and remap labels to binary (classifiable=0, trash=1)
2. Visualize the embedding space (PCA 2D)
3. Model selection: grid search over SVM configurations + PCA components
4. Cross-validation on training set (mean ± std)
5. Train final model on full training set
6. Evaluate on held-out test set
7. Save the trained model (scaler + PCA + SVM)

**Output files:**
- `model1_scaler.pkl` — StandardScaler fitted on training data
- `model1_pca.pkl` — PCA (50 components) fitted on training data
- `model1_svm.pkl` — trained SVM classifier

## 1. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import joblib
import warnings
warnings.filterwarnings('ignore')

from sklearn.svm import SVC
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import PCA
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.metrics import (
    accuracy_score, f1_score, precision_score, recall_score,
    confusion_matrix, classification_report, roc_auc_score
)
from sklearn.pipeline import Pipeline

## 2. Configuration

In [ ]:
# ── Input ──────────────────────────────────────────────────────────────────
EMBEDDINGS_PATH = "/content/drive/MyDrive/model_nuc/embeddings_radio.npz"

# ── Output ─────────────────────────────────────────────────────────────────
MODEL_DIR = "/content/drive/MyDrive/model_nuc"

# ── Label mapping — original 4 classes → binary ────────────────────────────
# 0=good, 1=invaginated, 2=Unclassifiable → classifiable (0)
# 3=trash                                 → trash (1)
LABEL_MAP = {0: 0, 1: 0, 2: 0, 3: 1}
CLASS_NAMES = {0: "classifiable", 1: "trash"}

# ── Reproducibility ────────────────────────────────────────────────────────
RANDOM_STATE = 42

## 3. Load Data & Prepare Binary Labels

In [ ]:
data   = np.load(EMBEDDINGS_PATH, allow_pickle=True)
X_all  = data["X"]
labels = data["labels"]
paths  = data["paths"]

# Remap to binary labels
y = np.array([LABEL_MAP[l] for l in labels])

print(f"Embedding shape: {X_all.shape}")
print(f"\nClass distribution:")
for cls, name in CLASS_NAMES.items():
    n = (y == cls).sum()
    print(f"  {name:>15}: {n:>4} samples ({100*n/len(y):.1f}%)")
print(f"  {'Total':>15}: {len(y):>4}")

## 4. Train / Test Split

20% held-out test set — not used until final evaluation.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X_all, y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y
)

print(f"Train: {len(X_train)} samples")
print(f"Test:  {len(X_test)} samples (held out)")

## 5. Embedding Space Visualization (PCA 2D)

Project the 2560-dim embeddings to 2D to check class separability
before training any classifier.

In [ ]:
scaler_viz = StandardScaler()
X_scaled   = scaler_viz.fit_transform(X_all)

pca_viz = PCA(n_components=2, random_state=RANDOM_STATE)
X_2d    = pca_viz.fit_transform(X_scaled)

print(f"Explained variance: PC1={pca_viz.explained_variance_ratio_[0]:.1%}  "
      f"PC2={pca_viz.explained_variance_ratio_[1]:.1%}")

COLORS = {0: "steelblue", 1: "tomato"}

plt.figure(figsize=(7, 6))
for cls, name in CLASS_NAMES.items():
    mask = y == cls
    plt.scatter(X_2d[mask, 0], X_2d[mask, 1],
                c=COLORS[cls], label=f"{name} (n={mask.sum()})",
                alpha=0.6, s=25, edgecolors="white", linewidth=0.3)

plt.title("PCA of RADIO Embeddings — Binary Labels")
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 6. Model Selection — Grid Search

Compare SVM configurations (kernel, C, PCA components) using 5-fold cross-validation
on the training set only.

The test set is not touched at this stage.

In [ ]:
# Configurations to compare: (name, sklearn Pipeline)
CONFIGS = {
    "SVM rbf  C=0.1":
        Pipeline([("scaler", StandardScaler()),
                  ("pca",    PCA(n_components=50)),
                  ("clf",    SVC(kernel="rbf", C=0.1,  class_weight="balanced", random_state=RANDOM_STATE))]),

    "SVM rbf  C=1.0":
        Pipeline([("scaler", StandardScaler()),
                  ("pca",    PCA(n_components=50)),
                  ("clf",    SVC(kernel="rbf", C=1.0,  class_weight="balanced", random_state=RANDOM_STATE))]),

    "SVM rbf  C=5.0":
        Pipeline([("scaler", StandardScaler()),
                  ("pca",    PCA(n_components=50)),
                  ("clf",    SVC(kernel="rbf", C=5.0,  class_weight="balanced", random_state=RANDOM_STATE))]),

    "SVM rbf  C=1 PCA100":
        Pipeline([("scaler", StandardScaler()),
                  ("pca",    PCA(n_components=100)),
                  ("clf",    SVC(kernel="rbf", C=1.0,  class_weight="balanced", random_state=RANDOM_STATE))]),

    "SVM lin  C=1.0":
        Pipeline([("scaler", StandardScaler()),
                  ("pca",    PCA(n_components=50)),
                  ("clf",    SVC(kernel="linear", C=1.0, class_weight="balanced", random_state=RANDOM_STATE))]),
}

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

grid_results = {}

print(f"{'Config':<25} {'F1 macro':>14} {'Accuracy':>12} {'Recall trash':>14}")
print("-" * 70)

best_name, best_f1 = None, 0

for name, pipeline in CONFIGS.items():
    cv_f1, cv_acc, cv_conf = [], [], []

    for train_idx, val_idx in skf.split(X_train, y_train):
        X_tr, X_val = X_train[train_idx], X_train[val_idx]
        y_tr, y_val = y_train[train_idx], y_train[val_idx]

        pipeline.fit(X_tr, y_tr)
        y_pred = pipeline.predict(X_val)

        cv_f1.append(f1_score(y_val, y_pred, average="macro"))
        cv_acc.append(accuracy_score(y_val, y_pred))
        cv_conf.append(confusion_matrix(y_val, y_pred, labels=[0, 1]))

    f1_arr   = np.array(cv_f1)
    acc_arr  = np.array(cv_acc)
    mean_cm  = np.mean(cv_conf, axis=0)

    # Recall for trash = TP_trash / (TP_trash + FN_trash)
    recall_trash = mean_cm[1, 1] / mean_cm[1].sum() if mean_cm[1].sum() > 0 else 0

    grid_results[name] = {"f1": f1_arr, "acc": acc_arr, "mean_cm": mean_cm}

    print(f"{name:<25} {f1_arr.mean():.4f}±{f1_arr.std():.4f} "
          f"{acc_arr.mean():.4f}±{acc_arr.std():.4f} "
          f"{recall_trash:>12.4f}")

    if f1_arr.mean() > best_f1:
        best_f1   = f1_arr.mean()
        best_name = name

print(f"\nBest config: {best_name}  (CV F1 macro = {best_f1:.4f})")

## 7. Cross-Validation on Training Set

5-fold stratified cross-validation on the training set.
Reports mean ± std for all key metrics.

Architecture fixed from grid search: SVM (rbf kernel) + StandardScaler + PCA(50), C=1.0.

In [ ]:
from sklearn.model_selection import cross_validate
from sklearn.metrics import make_scorer
from sklearn.pipeline import Pipeline

# Fixed C from grid search
FINAL_C = 1.0

# Full pipeline: scaler → PCA → SVM
pipeline_cv = Pipeline([
    ("scaler", StandardScaler()),
    ("pca",    PCA(n_components=50, random_state=RANDOM_STATE)),
    ("clf",    SVC(kernel="rbf", C=FINAL_C, class_weight="balanced",
                  probability=True, random_state=RANDOM_STATE))
])

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

scoring = {
    "roc_auc":   "roc_auc",
    "f1_macro":  "f1_macro",
    "accuracy":  "accuracy",
    "recall_classifiable":  make_scorer(recall_score,    average=None, labels=[0]),
    "recall_trash":         make_scorer(recall_score,    average=None, labels=[1]),
    "precision_classifiable": make_scorer(precision_score, average=None, labels=[0]),
    "precision_trash":      make_scorer(precision_score, average=None, labels=[1]),
}

cv_results = cross_validate(
    pipeline_cv, X_train, y_train,
    cv=cv,
    scoring=scoring,
    return_train_score=False
)

print("=" * 50)
print("CROSS-VALIDATION RESULTS (5-fold, train set)")
print("=" * 50)
for metric in ["roc_auc", "f1_macro", "accuracy",
               "recall_classifiable", "recall_trash",
               "precision_classifiable", "precision_trash"]:
    vals = cv_results[f"test_{metric}"].flatten()
    print(f"  {metric:<28}: {vals.mean():.4f} ± {vals.std():.4f}")

## 8. Train Final Model & Evaluate on Test Set

Trained on the full training set, then evaluated once on the held-out test set.

In [ ]:
# Fit scaler and PCA on full training set
scaler_final = StandardScaler()
pca_final    = PCA(n_components=50, random_state=RANDOM_STATE)

X_train_scaled = pca_final.fit_transform(scaler_final.fit_transform(X_train))
X_test_scaled  = pca_final.transform(scaler_final.transform(X_test))

# Train final SVM
clf_final = SVC(kernel="rbf", C=FINAL_C, class_weight="balanced",
                probability=True, random_state=RANDOM_STATE)
clf_final.fit(X_train_scaled, y_train)

# ── Evaluate on held-out test set ─────────────────────────────────────────
y_test_pred  = clf_final.predict(X_test_scaled)
y_test_proba = clf_final.predict_proba(X_test_scaled)[:, 1]

print("=" * 55)
print("TEST SET RESULTS")
print("=" * 55)
print(f"  Accuracy:  {accuracy_score(y_test, y_test_pred):.4f}")
print(f"  F1 macro:  {f1_score(y_test, y_test_pred, average='macro'):.4f}")
print(f"  AUC:       {roc_auc_score(y_test, y_test_proba):.4f}")
print()
print(classification_report(y_test, y_test_pred,
                            target_names=["classifiable", "trash"]))

# Confusion matrix plot
cm = confusion_matrix(y_test, y_test_pred, labels=[0, 1])
fig, ax = plt.subplots(figsize=(5, 4))
im = ax.imshow(cm, cmap="Blues")
plt.colorbar(im, ax=ax)
ax.set_xticks([0, 1]); ax.set_yticks([0, 1])
ax.set_xticklabels(["classifiable", "trash"])
ax.set_yticklabels(["classifiable", "trash"])
ax.set_xlabel("Predicted")
ax.set_ylabel("True")
ax.set_title("Model 1 — Test Set Confusion Matrix")
for i in range(2):
    for j in range(2):
        ax.text(j, i, str(cm[i, j]), ha="center", va="center",
                color="white" if cm[i, j] > cm.max() / 2 else "black", fontsize=14)
plt.tight_layout()
plt.show()

## 9. Save Model

In [ ]:
# Save the three components needed for inference:
#   scaler → normalizes raw RADIO embeddings
#   pca    → reduces 2560-dim to 50-dim
#   svm    → binary classifier

joblib.dump(scaler_final, f"{MODEL_DIR}/model1_scaler.pkl")
joblib.dump(pca_final,    f"{MODEL_DIR}/model1_pca.pkl")
joblib.dump(clf_final,    f"{MODEL_DIR}/model1_svm.pkl")

print("✓ Model saved:")
print(f"  {MODEL_DIR}/model1_scaler.pkl")
print(f"  {MODEL_DIR}/model1_pca.pkl")
print(f"  {MODEL_DIR}/model1_svm.pkl")